In [2]:
import polars as pl

In [3]:
df = pl.read_csv("C:/Users/yaoze/Downloads/flights_sample_3m.csv",
                 infer_schema_length=10000,
                 ignore_errors=True)

# Step 1 — drop cancelled flights
df_arrived = df.filter(pl.col("CANCELLED") == 0)
print(df_arrived.shape)

(2920860, 32)


In [4]:
df_arrived = df_arrived.with_columns([
    pl.col("ARR_DELAY").clip(-90, 500),
    pl.col("DEP_DELAY").clip(-90, 500)
])

In [5]:
delay_cols = ["DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER", 
              "DELAY_DUE_NAS", "DELAY_DUE_SECURITY", 
              "DELAY_DUE_LATE_AIRCRAFT"]

df_arrived = df_arrived.with_columns([
    pl.col(c).fill_null(0) for c in delay_cols
])

In [6]:
df_arrived = df_arrived.with_columns([
    pl.when(pl.col("FL_DATE").str.starts_with("2020") | 
            pl.col("FL_DATE").str.starts_with("2021"))
    .then(1)
    .otherwise(0)
    .alias("COVID_FLAG")
])

print(df_arrived["COVID_FLAG"].value_counts())

shape: (2, 2)
┌────────────┬─────────┐
│ COVID_FLAG ┆ count   │
│ ---        ┆ ---     │
│ i32        ┆ u32     │
╞════════════╪═════════╡
│ 0          ┆ 1869166 │
│ 1          ┆ 1051694 │
└────────────┴─────────┘


In [8]:
import os
os.makedirs("../data/clean", exist_ok=True)

df_arrived.write_parquet("../data/clean/flights_clean.parquet")
print("saved:", df_arrived.shape)

saved: (2920860, 33)
